In [9]:
%pip install selenium pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

In [37]:
driver= webdriver.Chrome()

In [38]:

url="https://wuzzuf.net/a/Accounting-Finance-Jobs-in-Egypt"
driver.get(url)

In [39]:
job_links = WebDriverWait(driver, 15).until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, 'a[href*="/jobs/p/"]')
    )
)

print(f"Jobs found on this page: {len(job_links)}")

Jobs found on this page: 20


In [40]:
jobs = driver.find_elements(By.TAG_NAME, "article")

In [41]:
JOB_TYPES = {
    "Full Time",
    "Part Time",
    "Freelance / Project",
    "Internship",
    "Shift Based",
    "Work From Home"
}

rows = []
seen_links = set()

for job_link in job_links:
    apply_link = job_link.get_attribute("href")

    if not apply_link or apply_link in seen_links:
        continue

    seen_links.add(apply_link)

    # الوصول إلى كارت الوظيفة الذي يحتوي على كل بيانات الإعلان
    card = driver.execute_script("""
        let node = arguments[0];

        while (node && node !== document.body) {
            const jobs = node.querySelectorAll('a[href*="/jobs/p/"]');

            if (
                jobs.length === 1 &&
                /Egypt/.test(node.innerText) &&
                node.innerText.length < 3000
            ) {
                return node;
            }

            node = node.parentElement;
        }

        return arguments[0].parentElement;
    """, job_link)

    lines = [line.strip() for line in card.text.splitlines() if line.strip()]

    job_title = job_link.text.strip()

    company_links = card.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/jobs/careers/"]'
    )

    company = (
        company_links[0].text.strip()
        if company_links
        else "Confidential"
    )

    location = next(
        (line for line in lines if "Egypt" in line),
        ""
    )

    job_type = next(
        (line for line in lines if line in JOB_TYPES),
        ""
    )

    rows.append({
        "job_name": job_title,
        "job_title": job_title,
        "company": company.rstrip(" -"),
        "location": location,
        "job_type": job_type,
        "apply_link": apply_link
    })

In [42]:
df = pd.DataFrame(
    rows,
    columns=[
        "job_name",
        "job_title",
        "company",
        "location",
        "job_type",
        "apply_link"
    ]
)

df.to_csv(
    "wuzzuf_jobs_page_1.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved {len(df)} jobs to wuzzuf_jobs_page_1.csv")

df

Saved 20 jobs to wuzzuf_jobs_page_1.csv


,job_name,job_title,company,location,job_type,apply_link
0,Facilities Manager-Engineering Sector,Facilities Manager-Engineering Sector,ARX Developments,"ARX Developments - Cairo, Egypt",,https://wuzzuf.net/jobs/p/z4qsm2aqtzfa-facilit...
1,Bank Accountant,Bank Accountant,Confidential,"Confidential - Sheraton, Cairo, Egypt",,https://wuzzuf.net/jobs/p/mngxqj1jihhp-bank-ac...
2,Inventory Manager,Inventory Manager,Simplex,"Simplex - 10th of Ramadan City, Cairo, Egypt",,https://wuzzuf.net/jobs/p/wtzd6i5uaix5-invento...
3,Fleet Supervisor,Fleet Supervisor,Confidential,"Confidential - Sheraton, Cairo, Egypt",,https://wuzzuf.net/jobs/p/hyy1i2i4er6m-fleet-s...
4,Logistic Specialist,Logistic Specialist,Simplex,"Simplex - 10th of Ramadan City, Cairo, Egypt",,https://wuzzuf.net/jobs/p/mdbqhj0h4z47-logisti...
5,Warehouse Manager,Warehouse Manager,Confidential,"Confidential - New Cairo, Cairo, Egypt",,https://wuzzuf.net/jobs/p/oyammlfpxbve-warehou...
6,Senior Accounts Payable Specialist,Senior Accounts Payable Specialist,MENA For Contracting & Trading,"MENA For Contracting & Trading - San Stefano, ...",,https://wuzzuf.net/jobs/p/qrif36rsnylp-senior-...
7,Senior Receivables Accountant,Senior Receivables Accountant,MENA For Contracting & Trading,"MENA For Contracting & Trading - San Stefano, ...",,https://wuzzuf.net/jobs/p/euu7nxugksq3-senior-...
8,Internal Audit Section Head,Internal Audit Section Head,Dr.Greiche,"Dr.Greiche - 10th of Ramadan City, Sharqia, Egypt",,https://wuzzuf.net/jobs/p/dowtggfbsphf-interna...
9,HR Manager / Generalist,HR Manager / Generalist,A1 arena,"A1 arena - Obour City, Cairo, Egypt",,https://wuzzuf.net/jobs/p/t8cqmrgr0mzl-hr-mana...


In [ ]:
driver.quit()